# Lesson 4.9: AI-Assisted Python

**Analyst:** Meron Welderufael
**Date:** 11/10/2026
**Notebook purpose:** Practice the four AI use patterns and the three-check
verification protocol on a small FinFlow analysis. Build the AI Use Note
habit before starting the Module 4 Project.

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

df = pd.read_csv(
    'finflow-enriched-200k.csv',
    parse_dates=['signup_date', 'last_login_date', 'churn_date']
)

print(f"Loaded {len(df):,} rows")
print(f"Columns: {list(df.columns)}")

Loaded 200,000 rows
Columns: ['customer_id', 'first_name', 'last_name', 'email', 'city', 'state', 'country', 'signup_date', 'plan_tier', 'monthly_fee', 'last_login_date', 'total_sessions', 'avg_session_duration_minutes', 'support_tickets_opened', 'active', 'churn_date', 'referral_source', 'marketing_name', 'target_segment', 'monthly_fee_listed', 'launch_date', 'plan_tier_clean']


## Pattern 1 in action, scaffolding new code
I have a Pandas DataFrame df with columns including marketing_name (string), signup_date (datetime), support_tickets_opened (int), and active (boolean). For customers where active is False (churned customers), compute the average support_tickets_opened grouped by marketing_name and by signup year. Return the result as a wide table with marketing_name as rows and signup year as columns. Explain what each step does.

In [2]:
# Step 1: filter to churned customers
churned = df[df['active'] == False].copy()

# Step 2: extract signup year
churned['signup_year'] = churned['signup_date'].dt.year

# Step 3: groupby and aggregate
result = churned.groupby(['marketing_name', 'signup_year'])['support_tickets_opened'].mean()

# Step 4: pivot to wide format with year as columns
result_wide = result.unstack()

print(result_wide)

signup_year      2024  2025  2026
marketing_name                   
Enterprise Plus  3.90  3.95  3.94
Professional     3.94  3.96  4.08
Starter Plan     4.22  4.00  4.04


In [3]:
#use small sample first
df_sample = df.head(500).copy()
churned = df_sample[df_sample['active'] == False].copy()
churned['signup_year'] = churned['signup_date'].dt.year

result = churned.groupby(['marketing_name', 'signup_year'])['support_tickets_opened'].mean()
result_wide = result.unstack()

print(result_wide)

signup_year     2024  2025  2026
marketing_name                  
Professional    5.00  3.73  3.50
Starter Plan    4.00  4.89  3.98


In [4]:
churned = df[df['active'] == False].copy()
churned['signup_year'] = churned['signup_date'].dt.year

result = churned.groupby(['marketing_name', 'signup_year'])['support_tickets_opened'].mean()
result_wide = result.unstack().round(2)

print(result_wide)

signup_year      2024  2025  2026
marketing_name                   
Enterprise Plus  3.90  3.95  3.94
Professional     3.94  3.96  4.08
Starter Plan     4.22  4.00  4.04


**Sanity check:** Average support tickets per churned customer per
(tier, year) sits close to 4.0 across all cells, the midpoint of the
0-to-8 draw in the Lesson 4.2 generation design. Wide-table shape is
3 rows by 3 columns as expected. Verification passes.

In [5]:
# This cell contains a deliberate error for practicing the debug workflow
fees_summary = churned.groupby('marketing_name').agg(
    total_fees=('monthly_fee', 'sum'),
    avg_sessions=('total_sessions', 'mean')
)
print(fees_summary)

                 total_fees  avg_sessions
marketing_name                           
Enterprise Plus   198602.00         25.53
Professional      266791.04         25.63
Starter Plan      300738.96         25.41


## AI Use Note

Used Claude throughout this lesson notebook for three patterns:

1. **Scaffolding (Step 5):** Asked Claude to write the churned-customers-by-tier-by-year analysis. Output was correct on the first try, including the `.copy()` defensive pattern from Lesson 4.5. Minor stylistic difference: AI used `df['active'] == False` where I'd write `~df['active']`. Kept AI's version.

2. **Debugging (Step 10):** Pasted a KeyError into Claude with the surrounding code. Claude correctly identified that `monthly_fees` should be `monthly_fee` (singular) based on the typical Pandas column-naming convention for fee data.

3. **API conversion (Step 11):** Asked Claude to convert a Pandas snippet to Polars. Output matched the expression-based Polars syntax from Lesson 4.8. Didn't run the Polars version because the lesson's focus was the conversion pattern itself.

Verification protocol (read-before-run, small-data-first, sanity-check-output) applied at each AI-generated cell. No AI suggestions silently shipped to the analysis. Final notebook is my work, AI-assisted.